<a href="https://colab.research.google.com/github/vikranthrach/IIT-Patna--AI-and-ML-Course/blob/main/Structured_tool_call.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
def calculate_sum(a,b):
  return int(a) + int(b)

print(calculate_sum(3, 5))

8


In [ ]:
print(calculate_sum("three", "five"))

ValueError: invalid literal for int() with base 10: 'three'

# Import modules

In [ ]:
import json
import os
from openai import OpenAI
from dotenv import load_dotenv
from google.colab import userdata
openaiapikey = userdata.get('OPENAI_API_KEY')
client = OpenAI(api_key=openaiapikey)

print("✅ Setup complete!")

✅ Setup complete!


In [ ]:
# Ask the same question 3 times — notice the format changes each time

prompt = ''' Summarize this customer request and identify what they need:
            'Hi, I was looking at your website and I think I want to get
            some of those Samsung phones for my team -
             probably around 3 of them, and I saw they
              were listed at around 25k each,
              but I'm not sure if that includes GST or not.
               Can you also check if there's a bulk discount?' '''

print("Asking the SAME question 3 times (with temperature=1 for variety):")
print("=" * 60)

for i in range(3):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=1.0
    )
    print(f"\nAttempt {i+1}:")
    print(response.choices[0].message.content)
    print("-" * 40)

Asking the SAME question 3 times (with temperature=1 for variety):

Attempt 1:
The customer is interested in purchasing three Samsung phones for their team and is inquiring about the total price, specifically whether the listed price of 25,000 each includes GST. Additionally, they want to know if there is a bulk discount available for their order.
----------------------------------------

Attempt 2:
The customer is interested in purchasing three Samsung phones for their team and wants to confirm the price of approximately 25,000 each, specifically whether that price includes GST. They also want to know if a bulk discount is available.
----------------------------------------

Attempt 3:
The customer is interested in purchasing approximately three Samsung phones for their team. They want to confirm the price of 25,000 each and whether it includes GST. Additionally, they are inquiring about the availability of a bulk discount for their purchase.
----------------------------------------


In [ ]:
messages=[{"role": "system", "content": "You are an extract product into. Always respond in JSON format with keys: product, price, quantity"},
          {"role": "user", "content": "I want to buy 3 sansung Galazy phones at rs 25000 each."}]

response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=2.0,
        response_format = {"type": "json_object"}) # This is forcing LLM to respons in a json output.

print(response.choices[0].message.content)

{
  "product": "Samsung Galaxy Phone",
  "price": 25000,
  "quantity": 3
}


# JSON Schema comes into picture

In [ ]:
response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        temperature=2.0,
        response_format = {
            "type" : "json_schema",
            "json_schema" : {
                "name": "product_extracton",
                "strict" :  True,
                "schema" : {
                    "type" : "object",
                    "properties" : {
                        "product_name" : {
                            "type" : "string",
                            "description" : "Name of the Product"
                        },
                        "price_per_unit": {
                            "type": "number",
                            "description" : "Price per one unit in INR"
                                                    },
                        "quantity" : {
                            "type" : "integer",
                            "description": "Number of units requested"
                        },
                        "total_amount" : {
                            "type" : "number",
                            "description" : "Total cost calcualted via (price_per_unit * quantity)"
                        }
                    },
                    "required" : ["product_name", "price_per_unit", "quantity", "total_amount"],
                    "additionalProperties" : False
                }
            }
        })

result = json.loads(response.choices[0].message.content)
print(json.dumps(result, indent = 4))

{
    "product_name": "samsung Galazy",
    "price_per_unit": 25000,
    "quantity": 3,
    "total_amount": 75000
}


In [ ]:
texts = [
    "I want to buy 3 samsung Galaxy phones at  twenty five thousand rupees each",
    "Please oder 10 noteboooks for 50 rupees per piece",
    "Get me 2 Macbook pro laptops, each costing 1.5 lakhs."
]

print("Extracting From 3 different Texts - same schema every time!!")




print("+" * 60)

for text in texts:
  response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are an extract product into. Always respond in JSON format with keys: product, price, quantity"},
            {"role": "user", "content": text}],
        temperature=2.0,
        response_format = {
            "type" : "json_schema",
            "json_schema" : {
                "name": "product_extracton",
                "strict" :  True,
                "schema" : {
                    "type" : "object",
                    "properties" : {
                        "product_name" : {
                            "type" : "string",
                            "description" : "Name of the Product"
                        },
                        "price_per_unit": {
                            "type": "number",
                            "description" : "Price per one unit in INR"
                                                    },
                        "quantity" : {
                            "type" : "integer",
                            "description": "Number of units requested"
                        },
                        "total_amount" : {
                            "type" : "number",
                            "description" : "Total cost calcualted via (price_per_unit * quantity)"
                        }
                    },
                    "required" : ["product_name", "price_per_unit", "quantity", "total_amount"],
                    "additionalProperties" : False
                }
            }
        })

  result = json.loads(response.choices[0].message.content)
  print(result)



Extracting From 3 different Texts - same schema every time!!
++++++++++++++++++++++++++++++++++++++++++++++++++++++++++++
{'product_name': 'Samsung Galaxy Phone', 'price_per_unit': 25000, 'quantity': 3, 'total_amount': 75000}
{'product_name': 'Notebook', 'price_per_unit': 50, 'quantity': 10, 'total_amount': 500}
{'product_name': 'Macbook Pro', 'price_per_unit': 150000, 'quantity': 2, 'total_amount': 300000}


In [ ]:
# When to use Json Object and When to use Json Schema

# Validaiton

-- Even with strucutred outputs, things can go wrong.
-- A required firled might be null.
-- Json migh be syntactically correct, but logically wrong.

LLM Response --> Parse this Json ---> Validatate this sjson --> You will use the output or trust the output for downstream jobs.

# Build a simple Json VALIDATOR.

In [ ]:
isinstance("Balaji", int)

False

In [ ]:
def validate_product_response(raw_response):
  try:
    data = json.loads(raw_response)
  except json.JSONDecodeError as e:
    return False, f"Not a valid JSON: {e}"

  required_keys = ["product_name", "price_per_unit", "quantity", "total_amount"]
  missing = [k for k in required_keys if k not in data]
  if missing:
    return False, f"Missing Keys: {missing}"

  if not isinstance(data['product_name'], str):
    return False, "Product Name must be string"

  if not isinstance(data['price_per_unit'], (int, float)):
    return False, "price_per_unit must be integer or float"

  if not isinstance(data['quantity'], int):
    return False, "quantity  must be Integer"

  if not isinstance(data['total_amount'], (int, float)):
    return False, "total_amount  must be intezer or float"

  if data['price_per_unit'] <= 0:
    return False, "price_per_unit must be greater than zero"

  if data['quantity'] <= 0:
    return False , "Quantity can never be negative"

  return True, data



# Test with good data
good = '{"product_name": "Samsung Galaxy", "price_per_unit": 25000, "quantity": 3, "total_amount": 75000}'
is_valid, result = validate_product_response(good)
print(f"✅ Good data: valid={is_valid}, result={result}")


✅ Good data: valid=True, result={'product_name': 'Samsung Galaxy', 'price_per_unit': 25000, 'quantity': 3, 'total_amount': 75000}


In [ ]:
# Test with bad data
bad_json = "This is not JSON at all"
is_valid, result = validate_product_response(bad_json)
print(f"❌ Bad JSON:  valid={is_valid}, error={result}")

❌ Bad JSON:  valid=False, error=Not a valid JSON: Expecting value: line 1 column 1 (char 0)


In [ ]:
# Test with missing key
missing = '{"product_name": "Phone", "price_per_unit": 25000}'
is_valid, result = validate_product_response(missing)
print(f"❌ Missing:   valid={is_valid}, error={result}")

❌ Missing:   valid=False, error=Missing Keys: ['quantity', 'total_amount']


In [ ]:
# Test with bad value
bad_value = '{"product_name": "Phone", "price_per_unit": -500, "quantity": 3, "total_amount": -1500}'
is_valid, result = validate_product_response(bad_value)
print(f"❌ Bad value: valid={is_valid}, error={result}")

❌ Bad value: valid=False, error=price_per_unit must be greater than zero


In [ ]:
# ──────────────────────────────────────
# Combining: Structured Output + Validation + Retry
# ──────────────────────────────────────

def extract_product_info(text, max_retries=3):
    """
    Extract product info from text with:
    1. Structured output (forces JSON)
    2. Validation (checks correctness)
    3. Retry (tries again on failure)
    """
    for attempt in range(max_retries):
        # Call LLM with structured output
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "Extract product information from the text."},
                {"role": "user", "content": text}
            ],
            response_format={
                "type": "json_schema",
                "json_schema": {
                    "name": "product_extraction",
                    "strict": True,
                    "schema": {
                        "type": "object",
                        "properties": {
                            "product_name": {"type": "string"},
                            "price_per_unit": {"type": "number"},
                            "quantity": {"type": "integer"},
                            "total_amount": {"type": "number"}
                        },
                        "required": ["product_name", "price_per_unit", "quantity", "total_amount"],
                        "additionalProperties": False
                    }
                }
            }
        )

        raw = response.choices[0].message.content

        # Validate
        is_valid, result = validate_product_response(raw)

        if is_valid:
            return result  # ✅ Success
        else:
            print(f"  ⚠️ Attempt {attempt + 1} failed validation: {result}")

    return None  # All retries failed

# Test
info = extract_product_info("I need 5 HP printers at 15000 each")
print("Extracted:")
print(json.dumps(info, indent=2))

Extracted:
{
  "product_name": "HP Printer",
  "price_per_unit": 15000,
  "quantity": 5,
  "total_amount": 75000
}


In [ ]:
# Call LLM - Parse Json - Validate - Use ( RETRY )

In [ ]:
def get_weather(city):
  return "32 degreed , parlty cloudy" # Unstructred output.


def get_weather(city):
  return json.dumps({"city": city, "temp_c": 32, "condition": "partly clody"}) # This is a structred output and it si reliable.

# CSV Agent Tool

In [ ]:
# ──────────────────────────────────────
# Upgraded tools with structured responses
# ──────────────────────────────────────

# Employee data (same as Session 04)
import csv
from io import StringIO

CSV_DATA = """name,department,salary,experience_years,city
Rahul,Engineering,75000,5,Hyderabad
Priya,Marketing,55000,3,Bangalore
Arjun,Engineering,82000,7,Hyderabad
Sneha,HR,48000,2,Mumbai
Vikram,Engineering,90000,9,Delhi
Anita,Marketing,60000,4,Bangalore
Karthik,HR,52000,3,Chennai
Deepa,Engineering,70000,4,Hyderabad
Suresh,Marketing,65000,6,Mumbai
Kavya,HR,50000,2,Bangalore
Ravi,Engineering,95000,10,Delhi
Meera,Marketing,58000,3,Chennai
Arun,Engineering,78000,6,Hyderabad
Lavanya,HR,55000,4,Bangalore
Sanjay,Marketing,62000,5,Mumbai
"""

reader = csv.DictReader(StringIO(CSV_DATA.strip()))
employees = list(reader)

# ─── Better tools with structured output ───

def get_stats(column: str) -> str:
    """Get stats for a column. Returns structured JSON."""
    try:
        values = [float(row[column]) for row in employees]
        return json.dumps({
            "status": "success",
            "column": column,
            "count": len(values),
            "mean": round(sum(values) / len(values), 2),
            "min": min(values),
            "max": max(values)
        })
    except (ValueError, KeyError):
        return json.dumps({
            "status": "error",
            "message": f"Column '{column}' is not numeric or doesn't exist",
            "available_numeric_columns": ["salary", "experience_years"]
        })

def filter_data(column: str, value: str) -> str:
    """Filter rows and return structured results."""
    try:
        matches = [row for row in employees if row[column].lower() == value.lower()]
        return json.dumps({
            "status": "success",
            "filter": {"column": column, "value": value},
            "count": len(matches),
            "results": matches
        })
    except KeyError:
        return json.dumps({
            "status": "error",
            "message": f"Column '{column}' not found",
            "available_columns": list(employees[0].keys())
        })

# Compare: old style vs new style
print("── Structured tool response: ──")
print(get_stats("salary"))
print()
print("── Error response (also structured): ──")
print(get_stats("age"))  # column doesn't exist

── Structured tool response: ──
{"status": "success", "column": "salary", "count": 15, "mean": 66333.33, "min": 48000.0, "max": 95000.0}

── Error response (also structured): ──
{"status": "error", "message": "Column 'age' is not numeric or doesn't exist", "available_numeric_columns": ["salary", "experience_years"]}


# Tool Idempotency In LLM

In [ ]:
# Non Idempotent Tool Call

orders_bad = []
def place_order_bad(product: str, quantity: int):
  order = {'product': product, "quanitty": quantity, "id": len(orders_bad) +1}
  orders_bad.append(order)
  return json.dumps({"status": "success", "order": order})

print(place_order_bad("Laptop", 1))
print(place_order_bad("Laptop", 1))

print(len(orders_bad))

{"status": "success", "order": {"product": "Laptop", "quanitty": 1, "id": 1}}
{"status": "success", "order": {"product": "Laptop", "quanitty": 1, "id": 2}}
2


In [ ]:
orders_good = {}

def place_order_safe(product: str, quantity: str, request_id: str):

  if request_id in orders_good:
    return json.dumps({
        "status": "success",
        "message": "Already processed ( duplicate tool call)",
        "order": orders_good[request_id]
    })

  order = {"product": product, "quantity": quantity, "request_id": request_id}
  orders_good['request_id'] = order
  return json.dumps({"status": "success", "order": order})

# Simulate agent calling it twice with same request_id
print("✅ IDEMPOTENT tool (called twice with same request_id):")
print(place_order_safe("Laptop", 1, request_id="order_abc123"))
print(place_order_safe("Laptop", 1, request_id="order_abc123"))  # Same ID = no duplicate
print(f"   Orders in system: {len(orders_good)} ← Correctly just 1! ✅")

✅ IDEMPOTENT tool (called twice with same request_id):
{"status": "success", "order": {"product": "Laptop", "quantity": 1, "request_id": "order_abc123"}}
{"status": "success", "order": {"product": "Laptop", "quantity": 1, "request_id": "order_abc123"}}
   Orders in system: 1 ← Correctly just 1! ✅


# Updating CSV agent for robustness

In [ ]:
# ──────────────────────────────────────
# Tool definitions (JSON schemas)
# ──────────────────────────────────────

report_tools = [
    {
        "type": "function",
        "function": {
            "name": "get_stats",
            "description": "Get statistics (mean, min, max, count) for a numeric column. Available columns: salary, experience_years.",
            "parameters": {
                "type": "object",
                "properties": {
                    "column": {"type": "string", "description": "Numeric column name"}
                },
                "required": ["column"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "filter_data",
            "description": "Filter employees by a column value. Available columns: name, department, city. Returns matching employees.",
            "parameters": {
                "type": "object",
                "properties": {
                    "column": {"type": "string", "description": "Column to filter on"},
                    "value": {"type": "string", "description": "Value to match"}
                },
                "required": ["column", "value"]
            }
        }
    }
]

report_available_tools = {
    "get_stats": get_stats,
    "filter_data": filter_data,
}

print("✅ Report tools ready")

✅ Report tools ready


In [ ]:
# ──────────────────────────────────────
# The Reliable Report Agent
# ──────────────────────────────────────

def generate_report(question, verbose=True):
    """
    Ask a question → Agent uses tools → Returns a STRUCTURED report.

    The final output is always:
    {
        "question": "...",
        "answer": "...",
        "data_used": [...],
        "confidence": "high" | "medium" | "low"
    }
    """

    # Phase 1: Let agent gather data using tools
    messages = [
        {"role": "system", "content": (
            "You are a data analyst. Use tools to gather data, then answer the question. "
            "Use tools first before answering."
        )},
        {"role": "user", "content": question}
    ]

    if verbose:
        print(f"\n{'═' * 60}")
        print(f"📊 Question: {question}")
        print(f"{'─' * 60}")

    # Agent loop (gather data)
    max_steps = 5
    tool_results = []

    for step in range(max_steps):
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=report_tools,
        )

        choice = response.choices[0]

        if choice.finish_reason == "stop":
            break

        if choice.message.tool_calls:
            messages.append(choice.message)
            for tool_call in choice.message.tool_calls:
                func_name = tool_call.function.name
                args = json.loads(tool_call.function.arguments)

                if verbose:
                    print(f"  🔧 {func_name}({args})")

                result = report_available_tools[func_name](**args)
                tool_results.append({"tool": func_name, "args": args, "result": result})

                if verbose:
                    print(f"     → {result[:80]}..." if len(result) > 80 else f"     → {result}")

                messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": result
                })

    # Phase 2: Generate structured report from gathered data
    if verbose:
        print(f"{'─' * 60}")
        print("  📝 Generating structured report...")

    report_response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Generate a structured report based on the data analysis."},
            {"role": "user", "content": f"Question: {question}\n\nData gathered: {json.dumps(tool_results)}\n\nGenerate a report."}
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "analysis_report",
                "strict": True,
                "schema": {
                    "type": "object",
                    "properties": {
                        "question": {"type": "string"},
                        "answer": {"type": "string"},
                        "key_numbers": {
                            "type": "array",
                            "items": {"type": "string"}
                        },
                        "confidence": {
                            "type": "string",
                            "description": "high, medium, or low"
                        }
                    },
                    "required": ["question", "answer", "key_numbers", "confidence"],
                    "additionalProperties": False
                }
            }
        }
    )


    report = json.loads(report_response.choices[0].message.content)
    return report

print("✅ Report agent ready!")

✅ Report agent ready!


In [ ]:
# Test the report agent
report = generate_report("What's the salary situation in the Engineering department?")

print("\n📋 STRUCTURED REPORT:")
print(json.dumps(report, indent=2))


════════════════════════════════════════════════════════════
📊 Question: What's the salary situation in the Engineering department?
────────────────────────────────────────────────────────────
  🔧 filter_data({'column': 'department', 'value': 'Engineering'})
     → {"status": "success", "filter": {"column": "department", "value": "Engineering"}...
  🔧 get_stats({'column': 'salary'})
     → {"status": "success", "column": "salary", "count": 15, "mean": 66333.33, "min": ...
────────────────────────────────────────────────────────────
  📝 Generating structured report...

📋 STRUCTURED REPORT:
{
  "question": "What's the salary situation in the Engineering department?",
  "answer": "The Engineering department has a total of 6 employees with varying salaries based on their experience and location. The average salary for this department is higher than the overall average salary in the organization.",
  "key_numbers": [
    "Total Employees: 6",
    "Minimum Salary: $70,000",
    "Maximum Sal

In [ ]:
# Another question
report = generate_report("Compare the average experience between departments")

print("\n📋 STRUCTURED REPORT:")
print(json.dumps(report, indent=2))

# Access specific fields programmatically
print(f"\n── Programmatic access (no parsing needed): ──")
print(f"Answer: {report['answer']}")
print(f"Confidence: {report['confidence']}")
print(f"Key numbers: {report['key_numbers']}")


════════════════════════════════════════════════════════════
📊 Question: Compare the average experience between departments
────────────────────────────────────────────────────────────
  🔧 filter_data({'column': 'department', 'value': 'Sales'})
     → {"status": "success", "filter": {"column": "department", "value": "Sales"}, "cou...
  🔧 filter_data({'column': 'department', 'value': 'HR'})
     → {"status": "success", "filter": {"column": "department", "value": "HR"}, "count"...
  🔧 filter_data({'column': 'department', 'value': 'Engineering'})
     → {"status": "success", "filter": {"column": "department", "value": "Engineering"}...
  🔧 filter_data({'column': 'department', 'value': 'Marketing'})
     → {"status": "success", "filter": {"column": "department", "value": "Marketing"}, ...
  🔧 get_stats({'column': 'experience_years'})
     → {"status": "success", "column": "experience_years", "count": 15, "mean": 4.87, "...
  🔧 get_stats({'column': 'experience_years'})
     → {"status": "s

In [ ]:
# One more — showing multi-step tool usage
report = generate_report("How many people are in Hyderabad and what's their average salary?")

print("\n📋 STRUCTURED REPORT:")
print(json.dumps(report, indent=2))


════════════════════════════════════════════════════════════
📊 Question: How many people are in Hyderabad and what's their average salary?
────────────────────────────────────────────────────────────
  🔧 filter_data({'column': 'city', 'value': 'Hyderabad'})
     → {"status": "success", "filter": {"column": "city", "value": "Hyderabad"}, "count...
  🔧 get_stats({'column': 'salary'})
     → {"status": "success", "column": "salary", "count": 15, "mean": 66333.33, "min": ...
────────────────────────────────────────────────────────────
  📝 Generating structured report...

📋 STRUCTURED REPORT:
{
  "question": "How many people are in Hyderabad and what's their average salary?",
  "answer": "There are 4 individuals identified in Hyderabad, with an average salary of \u20b977,500.",
  "key_numbers": [
    "Number of people in Hyderabad: 4",
    "Average salary: \u20b977,500",
    "Minimum salary: \u20b970,000",
    "Maximum salary: \u20b982,000"
  ],
  "confidence": "high"
}
